In [ ]:
import requests
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("EIA_API_KEY")

SERIES = {
    "stocks": "PET.WCESTUS1.W",
    "production": "PET.WCRFPUS2.W",
    "imports": "PET.WCRIMUS2.W",
    "exports": "PET.WCREXUS2.W",
    "refinery_runs": "PET.WCRRIUS2.W",
}

In [19]:
def fetch_eia_data(series_id):
    url = f"https://api.eia.gov/v2/seriesid/{series_id}?api_key={API_KEY}"

    response = requests.get(url)
    if response.status_code != 200:
        raise Exception(f"API request failed: {response.text}")
    
    data = response.json()
    try:
        records = data["response"]["data"]
    except KeyError:
        raise Exception(f"Unexpected API response: {data}")
    df = pd.DataFrame(records)
    #'''
    df = df.rename(columns={
        "period": "date",
        "value": "value"
    })

    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    return df[["date", "value"]].dropna()
    #'''
    #return df
for name,series_id in SERIES.items():
        print(series_id)
        print(f"\nFetching {name}...")
        display(fetch_eia_data(series_id))  

PET.WCESTUS1.W

Fetching stocks...


,date,value
0,2026-03-27,461636
1,2026-03-20,456185
2,2026-03-13,449259
3,2026-03-06,443103
4,2026-02-27,439279
...,...,...
2265,1982-10-08,335260
2266,1982-10-01,334786
2267,1982-09-24,335586
2268,1982-08-27,336138


PET.WCRFPUS2.W

Fetching production...


,date,value
0,2026-03-27,13657
1,2026-03-20,13657
2,2026-03-13,13668
3,2026-03-06,13678
4,2026-02-27,13696
...,...,...
2248,1983-02-04,8660
2249,1983-01-28,8634
2250,1983-01-21,8634
2251,1983-01-14,8634


PET.WCRIMUS2.W

Fetching imports...


,date,value
0,2026-03-27,6454
1,2026-03-20,6464
2,2026-03-13,7194
3,2026-03-06,6422
4,2026-02-27,6324
...,...,...
1886,1990-02-02,6761
1887,1990-01-26,6144
1888,1990-01-19,6463
1889,1990-01-12,6644


PET.WCREXUS2.W

Fetching exports...


,date,value
0,2026-03-27,3521
1,2026-03-20,3322
2,2026-03-13,4898
3,2026-03-06,3434
4,2026-02-27,3997
...,...,...
1829,1991-03-08,167
1830,1991-03-01,242
1831,1991-02-22,242
1832,1991-02-15,138


PET.WCRRIUS2.W

Fetching refinery_runs...


,date,value
0,2026-03-27,16379
1,2026-03-20,16598
2,2026-03-13,16232
3,2026-03-06,16169
4,2026-02-27,15841
...,...,...
2265,1982-10-08,12062
2266,1982-10-01,12303
2267,1982-09-24,12375
2268,1982-08-27,11918


In [ ]:
def fetch_all():
    dfs = []

    for name, series_id in SERIES.items():
        print(f"Fetching {name}...")
        df = fetch_eia_data(series_id)
        df = df.rename(columns={"value": name})
        dfs.append(df)

    # Merge all series
    df_merged = dfs[0]

    for df in dfs[1:]:
        df_merged = df_merged.merge(df, on="date", how="outer")

    df_merged = df_merged.sort_values("date")
    df_merged = df_merged.set_index("date").ffill()
    df_merged["inventory_change"] = df_merged["stocks"].diff()


    return df_merged

display(fetch_all())

Fetching stocks...
Fetching production...
Fetching imports...
Fetching exports...
Fetching refinery_runs...


,stocks,production,imports,exports,refinery_runs
date,,,,,
1982-08-20,338764,NaN,NaN,NaN,11722
1982-08-27,336138,NaN,NaN,NaN,11918
1982-09-24,335586,NaN,NaN,NaN,12375
1982-10-01,334786,NaN,NaN,NaN,12303
1982-10-08,335260,NaN,NaN,NaN,12062
...,...,...,...,...,...
2026-02-27,439279,13696.0,6324.0,3997.0,15841
2026-03-06,443103,13678.0,6422.0,3434.0,16169
2026-03-13,449259,13668.0,7194.0,4898.0,16232
